### import libraries

In [35]:
import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
from XLO_sim.XLO_sim import XLO_sim
from XLO_sim.Plot import XLO_plot
from XLO_sim import tools

import numpy.fft as fft

import scipy.constants as sp_const
au_in_eV = sp_const.value('atomic unit of energy') / sp_const.value('atomic unit of charge')

import multiprocessing as mp
import shutil

notebook_path = os.path.abspath("__file__")
notebook_directory = os.path.dirname(notebook_path)
base_directory = os.path.dirname(notebook_directory)

### Transmission

In [ ]:
tpad = 1000
xpad = 64
ypad = 64

hwKalpha1N = 8047.91

In [ ]:
def yaml_modify_seed_energy_and_target_energy(input_yaml_path, output_yaml_path, new_seed_energy, target_energy_eV):
    # Read the original YAML file
    with open(input_yaml_path, 'r') as file:
        yaml_data = yaml.safe_load(file)

    yaml_data['E_seed_uJ'] = new_seed_energy
    yaml_data['monochromator_target_energy_eV'] = float(target_energy_eV)

    # Write the modified data to a new YAML file
    with open(output_yaml_path, 'w') as file:
        yaml.safe_dump(yaml_data, file)

    print(f"Modified YAML file saved to {output_yaml_path}")

In [ ]:
ar_Eseed_values = [40] #[200, 150, 100, 70, 40, 20, 10, 5, 1, 0.1]

# Absolute target photon energies to scan (eV), spanning +/-15 eV around the
# Cu Kalpha1 line (hwKalpha1N, read from the base config below).
with open(base_directory + '/config/base/Cu-seed-mono-SASE.yaml', 'r') as file:
    hwKalpha1N = yaml.safe_load(file)['hwKalpha1N']
ar_energy_values = hwKalpha1N + np.arange(-15, 16, 3, dtype=float)

ar_yaml = []

for Eseed in ar_Eseed_values:
    generated_directory = base_directory + '/config/generated'
    generated_path = os.path.join(generated_directory)
    if not os.path.exists(generated_path):
        os.makedirs(generated_path)

    for target_energy_eV in ar_energy_values:
        input_yaml_path = base_directory + '/config/base/Cu-seed-mono-SASE.yaml'
        output_yaml_path = base_directory + '/config/generated/Cu-seed-mono-SASE_' + f'{Eseed:.2f}' + 'uJ_' + f'{target_energy_eV:.2f}' + 'eV.yaml'
        yaml_modify_seed_energy_and_target_energy(input_yaml_path, output_yaml_path, Eseed, target_energy_eV)
        ar_yaml.append(output_yaml_path)

### calculating for given amount of repetitions for each yaml

In [ ]:
nrep = 1

# Get the number of available CPU cores
num_cpus = mp.cpu_count()

# Set the maximum number of processes to use
num_processes = num_cpus - 1  # Leave one core for system processes or other tasks
print("num_processes", num_processes)


data_path = os.path.join(base_directory, 'data/' + np.datetime_as_string(np.datetime64('now')))
if not os.path.exists(data_path):
    os.makedirs(data_path)

def run_simulation(yaml, run_path, rep):
    print('repetition ', rep + 1)

    X = XLO_sim(yaml)
    X.random_seed = rep  # Set the random seed for reproducibility
    seed_field = tools.Ocelot_SASE_seed_111_dcm_pstxy(X)
    X.configure(seed_field)
    X.run_3D()

    womega_ar, I_int_thy_w_0, I_thy0_w_0 = tools.SF_spectrum_w(X, 0, ypad, tpad)
    womega_ar, I_int_thy_w_last, I_thy0_w_last = tools.SF_spectrum_w(X, -1, ypad, tpad)

    target_energy_eV = X.monochromator_target_energy_eV
    date_string = np.datetime_as_string(np.datetime64('now'))
    np.savez_compressed(
        os.path.join(run_path, f"run_at_seed_{X.E_seed_uJ:.1f}_uJ__energy_{target_energy_eV:.2f}_eV__repetition_{rep + 1}_{date_string}.npz"),
        target_energy_eV=target_energy_eV,
        womega_ar=womega_ar,
        I_int_thy_w_0=I_int_thy_w_0,
        I_thy0_w_0=I_thy0_w_0,
        I_int_thy_w_last=I_int_thy_w_last,
        I_thy0_w_last=I_thy0_w_last
    )

def run_repetitions_from_yaml(yaml, nrep, num_processes, data_path):
    date_string = np.datetime_as_string(np.datetime64('now'))
    X = XLO_sim(yaml)
    target_energy_eV = X.monochromator_target_energy_eV

    run_path = os.path.join(data_path, f'runs_seed_{X.E_seed_uJ:.1f}_uJ__energy_{target_energy_eV:.2f}_eV_{date_string}')
    if not os.path.exists(run_path):
        os.makedirs(run_path)

    shutil.copy2(yaml, run_path)

    # Force the 'fork' start method explicitly so behavior is consistent
    # between macOS (default 'spawn' since Python 3.8) and the Linux
    # cluster (default 'fork'). 'spawn' fails here with AttributeError
    # because it re-imports __main__ to find run_simulation, which does
    # not work for functions defined in a notebook.
    ctx = mp.get_context('fork')
    with ctx.Pool(processes=num_processes) as pool:
        pool.starmap(run_simulation, [(yaml, run_path, rep) for rep in range(nrep)])

if __name__ == '__main__':
    for yaml_value in ar_yaml:
        print(f"Running simulations for YAML file: {yaml_value}")
        run_repetitions_from_yaml(yaml_value, nrep, num_processes, data_path)

### reading and plotting

In [ ]:
figs_path = os.path.join(base_directory, 'figs')
if not os.path.exists(figs_path):
    os.makedirs(figs_path)

In [ ]:
from collections import OrderedDict


def data_from_folder(folder_path):

    # Accumulated arrays, nested by E_seed and then by absolute target
    # energy: accumulated_arrays[E_seed_value][energy_value][array_name] ->
    # array averaged over repetitions. Each runs_.../ folder corresponds to
    # exactly one (E_seed, target energy) config, since cell 5 gives every
    # pair its own YAML (matching scripts/generate_mono_sweep_configs.py).
    accumulated_arrays = {}

    for runs_folder in os.listdir(folder_path):
        if not os.path.isdir(os.path.join(folder_path, runs_folder)):
            continue  # Skip if it's not a directory
        # Specify the path to the folder containing the saved files
        runs_path = folder_path + '/' + runs_folder

        # Find the YAML file in the folder
        yaml_file = None
        for file_name in os.listdir(runs_path):
            if file_name.endswith('.yaml'):
                yaml_file = os.path.join(runs_path, file_name)
                break

        if yaml_file is None:
            print(f"No YAML file found in {runs_path}")
            continue

        # Read the YAML file to get the E_seed value and target energy
        with open(yaml_file, 'r') as f:
            yaml_data = yaml.safe_load(f)
            E_seed_value = yaml_data['E_seed_uJ']
            energy_value = yaml_data['monochromator_target_energy_eV']

            hwKalpha1N = yaml_data['hwKalpha1N']
            tgrid = yaml_data['tgrid']
            xgrid = yaml_data['xgrid']
            ygrid = yaml_data['ygrid']
            zgrid = yaml_data['zgrid']
            sigma1_Ka1_2p3 = yaml_data['sigma1_Ka1_2p3']
            sigma1_Ka1_other = yaml_data['sigma1_Ka1_other']
            seed_duration_FWHM_t = yaml_data['seed_duration_FWHM_t']

            aux_data = [tgrid, xgrid, ygrid, zgrid, sigma1_Ka1_2p3, sigma1_Ka1_other, seed_duration_FWHM_t, hwKalpha1N]

        if E_seed_value not in accumulated_arrays:
            accumulated_arrays[E_seed_value] = {}
        accumulated_arrays[E_seed_value][energy_value] = {}

        # Iterate over all files in the folder
        for file_name in os.listdir(runs_path):
            # Check if the file is a numpy savez_compressed file
            if file_name.endswith('.npz'):
                # Load the file
                file_path = os.path.join(runs_path, file_name)
                data = np.load(file_path)

                # Iterate over all arrays in the loaded file
                for array_name in data.files:
                    if array_name == 'target_energy_eV':
                        continue
                    # Accumulate the arrays by averaging them
                    if array_name in accumulated_arrays[E_seed_value][energy_value]:
                        accumulated_arrays[E_seed_value][energy_value][array_name] += data[array_name]
                    else:
                        accumulated_arrays[E_seed_value][energy_value][array_name] = data[array_name]

        # Divide each accumulated array by the number of files to get the average
        num_files = len([fn for fn in os.listdir(runs_path) if fn.endswith('.npz')])
        for array_name in accumulated_arrays[E_seed_value][energy_value]:
            accumulated_arrays[E_seed_value][energy_value][array_name] /= num_files

    # Sort by E_seed (descending) and, within each E_seed, by target energy (ascending)
    sorted_accumulated_arrays = OrderedDict(sorted(accumulated_arrays.items(), key=lambda x: x[0], reverse=True))
    for E_seed_value in sorted_accumulated_arrays:
        sorted_accumulated_arrays[E_seed_value] = OrderedDict(
            sorted(sorted_accumulated_arrays[E_seed_value].items(), key=lambda x: x[0])
        )

    return sorted_accumulated_arrays, aux_data

In [ ]:
sorted_accumulated_arrays, aux_data = data_from_folder(data_path)

tgrid, xgrid, ygrid, zgrid, sigma1_Ka1_2p3, sigma1_Ka1_other, seed_duration_FWHM_t, hwKalpha1N = aux_data

plt.figure()

str_pars_grid = " Nx = " + str(xgrid) + ", Ny = " + str(ygrid) + ", Nz = " + str(zgrid) + ", Nt = " + str(tgrid)
str_pars_calc = r'$\sigma_{2p}$ = ' + f'{1e7*sigma1_Ka1_2p3:.1f}' + ' kbarn' + \
r', $\sigma_{other}$ = ' + f'{1e7*sigma1_Ka1_other:.1f}' +' kbarn' + \
r', $t_{seed}^{FWHM}$ = ' + f'{seed_duration_FWHM_t:.1f}' +' fs'

plt.title('average over ' + str(nrep) + ' SASE realizations per point' + '\n'
+ str_pars_grid + '\n' + str_pars_calc
 )

E_seed_values = list(sorted_accumulated_arrays.keys())
norm = plt.matplotlib.colors.LogNorm(vmin=min(E_seed_values), vmax=max(E_seed_values))
cmap = plt.matplotlib.colormaps['viridis']

transmittance_dict = {}

for E_seed_value, energy_arrays in sorted_accumulated_arrays.items():
    energy_values = np.array(sorted(energy_arrays.keys()))
    T_values = np.zeros_like(energy_values)

    for i, energy_value in enumerate(energy_values):
        arrays = energy_arrays[energy_value]
        I_int_thy_w_last = arrays['I_int_thy_w_last']
        I_int_thy_w_0 = arrays['I_int_thy_w_0']

        # Transmittance of a monochromated shot: total transmitted over total
        # incident spectral intensity, integrated over the narrow passband
        # selected by the monochromator at this target energy.
        T_values[i] = np.sum(I_int_thy_w_last) / np.sum(I_int_thy_w_0)

    transmittance_dict[E_seed_value] = (energy_values, T_values)

    plt.scatter(energy_values, T_values, color=cmap(norm(E_seed_value)), label='T at ' + str(E_seed_value) + ' uJ')

plt.axvline(hwKalpha1N, color='gray', ls='--', lw=1, label=r'$E_{K\alpha 1}$' + f' = {hwKalpha1N:.2f} eV')
plt.ylabel(r'$T$')
plt.xlabel(r'$E$ (eV)')
plt.xlim(hwKalpha1N - 15, hwKalpha1N + 15)
plt.legend()
plt.grid(True)
plt.savefig(figs_path + '/mono_' + str_pars_calc + '.pdf')
plt.show()

In [ ]:
min_T = []
area_T = []
FWHM_T = []

for E_seed_value, (energy_values, T_values) in transmittance_dict.items():
    min_T.append(np.min(T_values))

    n_edge = 2
    T_base = 0.5 * (
        np.mean(T_values[:n_edge]) +
        np.mean(T_values[-n_edge:])
    )

    dip = T_base - T_values

    area_T.append(np.trapz(dip, energy_values))
    FWHM_T.append(tools.find_fwhm(energy_values, dip))